In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

PROCESSED_DIR = Path("../data/processed")
ALL_MATCHES_PATH = PROCESSED_DIR / "atp_matches_slams.csv"
BSCORE_MATCHES_PATH = PROCESSED_DIR / "atp_matches_with_bscore.csv"

all_matches = pd.read_csv(ALL_MATCHES_PATH)
bscore_matches = pd.read_csv(BSCORE_MATCHES_PATH)

all_matches["tourney_date"] = pd.to_datetime(all_matches["tourney_date"])
bscore_matches["tourney_date"] = pd.to_datetime(bscore_matches["tourney_date"])

round_order = {
    "R128": 1,
    "R64": 2,
    "R32": 3,
    "R16": 4,
    "QF": 5,
    "SF": 6,
    "F": 7
}

all_matches["round_order"] = all_matches["round"].map(round_order)
bscore_matches["round_order"] = bscore_matches["round"].map(round_order)

all_matches = all_matches.dropna(subset=["winner_name", "loser_name", "round_order"]).copy()
bscore_matches = bscore_matches.dropna(subset=["winner_name", "loser_name", "round_order"]).copy()

all_matches["round_order"] = all_matches["round_order"].astype(int)
bscore_matches["round_order"] = bscore_matches["round_order"].astype(int)

In [24]:
def decay_weight(match_dates, t, alpha_days = 365):
    age_days = (t - match_dates).dt.days.clip(lower=0)
    return 1 / (1 + age_days / alpha_days)

def build_opponent_sets(history):
    opponents = defaultdict(set)

    for w, l in history [["winner_name", "loser_name"]].itertuples(index=False):
        opponents[w].add(l)
        opponents[l].add(w)
    
    return opponents

def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def compute_local_hodge_intransitivity(
    history, player_a, player_b, t, opponent_sets, alpha_days = 365, 
    min_common_opponents = 2, min_pair_weight=0.0, eps=0.5
):
    common_opponents = opponent_sets[player_a].intersection(opponent_sets[player_b])

    if len(common_opponents) < min_common_opponents:
        return {
            "num_common_opponents": len(common_opponents),
            "num_local_edges": 0,
            "norm_transitive": 0.0,
            "norm_cyclic": 0.0,
            "I_raw": 0.0,
            "direct_evidence": 0.0,
            "context_evidence": 0.0,
            "I_star_direct": 0.0,
            "I_star_context": 0.0
        }

    nodes = set(common_opponents)
    nodes.add(player_a)
    nodes.add(player_b)

    local_history = history[
        history["winner_name"].isin(nodes) & history["loser_name"].isin(nodes)
    ].copy()

    if local_history.empty:
        return {
        "num_common_opponents": len(common_opponents),
        "num_local_edges": 0,
        "norm_transitive": 0.0,
        "norm_cyclic": 0.0,
        "I_raw": 0.0,
        "direct_evidence": 0.0,            
        "context_evidence": 0.0,
        "I_star_direct": 0.0,
        "I_star_context": 0.0            
        }

    local_history["decay_weight"] = decay_weight(local_history["tourney_date"], t, alpha_days)

    pair_stats = {}
    for winner, loser, weight in local_history[
        ["winner_name", "loser_name", "decay_weight"]
    ].itertuples(index=False, name=None):
        p1, p2 = sorted([winner, loser])
        key = (p1, p2)

        if key not in pair_stats:
            pair_stats[key] = {"p1_beats_p2": 0.0, "p2_beats_p1": 0.0}

        if winner == p1:
            pair_stats[key]["p1_beats_p2"] += weight
        else:
            pair_stats[key]["p2_beats_p1"] += weight 

    players = sorted(nodes)
    player_to_idx = {p: i for i, p in enumerate(players)}

    rows = []
    advantages = []
    weights = []

    direct_evidence = 0.0
    context_evidence = 0.0

    target_key = tuple(sorted([player_a, player_b]))

    for (p1, p2), stats in pair_stats.items():
        w12 = stats["p1_beats_p2"]
        w21 = stats["p2_beats_p1"]
        total = w12 + w21

        if total <= min_pair_weight:
            continue

        if (p1, p2) == target_key:
            direct_evidence = total

        context_evidence += total

        prob_p1_beats_p2 = (w12 + eps) / (total + 2 * eps)
        advantage = logit(prob_p1_beats_p2)

        row = np.zeros(len(players))
        row[player_to_idx[p1]] = 1.0
        row[player_to_idx[p2]] = -1.0

        rows.append(row)
        advantages.append(advantage)
        weights.append(total)

    if len(rows) < 3:
        return {
                "num_common_opponents": len(common_opponents),
                "num_local_edges": len(rows),
                "norm_transitive": 0.0,
                "norm_cyclic": 0.0,
                "I_raw": 0.0,
                "direct_evidence": direct_evidence,
                "context_evidence": context_evidence,
                "I_star_direct": 0.0,
                "I_star_context": 0.0
            }
        
    B = np.vstack(rows)
    y = np.array(advantages)
    w = np.array(weights)

    sqrt_w = np.sqrt(w)
    B_weighted = B * sqrt_w[:, None]
    y_weighted = y * sqrt_w

    theta, *_ = np.linalg.lstsq(B_weighted, y_weighted, rcond = None)

    transitive = B @ theta
    cyclic = y - transitive

    norm_transitive = np.sqrt(np.sum(w * transitive ** 2))
    norm_cyclic = np.sqrt(np.sum(w * cyclic ** 2))

    I_raw = (1 + norm_cyclic) / (1 + norm_transitive)

    return {
            "num_common_opponents": len(common_opponents),
            "num_local_edges": len(rows),
            "norm_transitive": norm_transitive,
            "norm_cyclic": norm_cyclic,
            "I_raw": I_raw,
            "direct_evidence": direct_evidence,
            "context_evidence": context_evidence,
            "I_star_direct": I_raw * np.sqrt(direct_evidence),
            "I_star_context": I_raw * np.sqrt(context_evidence)
        }





In [25]:
START_DATE = pd.Timestamp("2011-01-01")
ALPHA_DAYS = 365

history = all_matches[all_matches["tourney_date"] < START_DATE].copy()
future = all_matches[all_matches["tourney_date"] >= START_DATE].copy()   

print("Historical matches before 2011:", len(history))
print("Matches to score from 2011 onward:", len(future))


Historical matches before 2011: 5800
Matches to score from 2011 onward: 15423


In [26]:
rolling_blocks = (
    future[["tourney_date", "tourney_id", "round_order"]]
    .drop_duplicates()
    .sort_values(["tourney_date", "tourney_id", "round_order"])
    .reset_index(drop=True)
)

print("Number of rolling blocks:", len(rolling_blocks))

rolling_blocks.head()

Number of rolling blocks: 1451


,tourney_date,tourney_id,round_order
0,2011-01-17,2011-580,1
1,2011-01-17,2011-580,2
2,2011-01-17,2011-580,3
3,2011-01-17,2011-580,4
4,2011-01-17,2011-580,5


In [27]:
intransitivity_rows = []

for block in rolling_blocks.itertuples(index=False):

    t_block = block.tourney_date

    tourney_id = block.tourney_id
    round_order_block = block.round_order

    block_matches = future[
        (future["tourney_id"] == tourney_id) &
        (future["round_order"] == round_order_block)
    ].copy()

    opponent_sets = build_opponent_sets(history)

    for row in block_matches.itertuples(index=False):

        player_a = row.winner_name
        player_b = row.loser_name

        score = compute_local_hodge_intransitivity(
            history = history, 
            player_a = player_a, 
            player_b = player_b, 
            t = t_block, 
            opponent_sets = opponent_sets, 
            alpha_days= ALPHA_DAYS,
            min_common_opponents = 2,

            eps = 0.5
        )

        score.update({
            "tourney_id": row.tourney_id,
            "tourney_date": row.tourney_date,
            "round": row.round,
            "round_order": row.round_order,
            "surface": row.surface,
            "winner_name": row.winner_name,
            "loser_name": row.loser_name
        })

        intransitivity_rows.append(score)

    history = pd.concat(
        [history, block_matches],
        ignore_index=True
    )

intransitivity_df = pd.DataFrame(intransitivity_rows)

print("Computed intransitivity scores for:", len(intransitivity_df), "matches")

intransitivity_df.head()

Computed intransitivity scores for: 15423 matches


,num_common_opponents,num_local_edges,norm_transitive,norm_cyclic,I_raw,direct_evidence,context_evidence,I_star_direct,I_star_context,tourney_id,tourney_date,round,round_order,surface,winner_name,loser_name
0,9,36,3.921288,2.728113,0.757548,0.0,33.385475,0.0,4.377126,2011-580,2011-01-17,R128,1,Hard,Roger Federer,Lukas Lacko
1,1,0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,2011-580,2011-01-17,R128,1,Hard,Xavier Malisse,Pablo Andujar
2,2,4,0.890106,0.399558,0.740465,0.0,1.936490,0.0,1.030415,2011-580,2011-01-17,R128,1,Hard,Marcos Baghdatis,Grega Zemlja
3,17,115,4.336663,4.635493,1.055996,0.0,66.809878,0.0,8.631424,2011-580,2011-01-17,R128,1,Hard,Juan Martin del Potro,Dudi Sela
4,1,0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,2011-580,2011-01-17,R128,1,Hard,Pere Riba,Carsten Ball


In [28]:
print("Rows", len(intransitivity_df))
print("columns", intransitivity_df.columns.tolist())

important_cols = [
    "num_common_opponents",
    "num_local_edges",
    "norm_transitive",
    "norm_cyclic",
    "I_raw",
    "direct_evidence",
    "context_evidence",
    "I_star_direct",
    "I_star_context"
]

print(intransitivity_df[important_cols].isna().sum())

# Summary statistics.
intransitivity_df[important_cols].describe()

Rows 15423
columns ['num_common_opponents', 'num_local_edges', 'norm_transitive', 'norm_cyclic', 'I_raw', 'direct_evidence', 'context_evidence', 'I_star_direct', 'I_star_context', 'tourney_id', 'tourney_date', 'round', 'round_order', 'surface', 'winner_name', 'loser_name']
num_common_opponents    0
num_local_edges         0
norm_transitive         0
norm_cyclic             0
I_raw                   0
direct_evidence         0
context_evidence        0
I_star_direct           0
I_star_context          0
dtype: int64


,num_common_opponents,num_local_edges,norm_transitive,norm_cyclic,I_raw,direct_evidence,context_evidence,I_star_direct,I_star_context
count,15423.000000,15423.000000,15423.000000,15423.000000,15423.000000,15423.000000,15423.000000,15423.000000,15423.000000
mean,25.724956,402.112624,6.670242,6.706675,0.860164,0.336020,234.364138,0.347443,12.092668
std,25.054022,605.074268,5.373813,5.763717,0.377857,0.689141,309.825594,0.516404,10.734072
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,5.000000,18.000000,1.990251,1.649203,0.770614,0.000000,11.054277,0.000000,2.871135
50%,18.000000,130.000000,5.513084,5.387993,0.955597,0.000000,88.781533,0.000000,9.413985
75%,40.000000,540.500000,10.838130,10.957630,1.078113,0.464377,355.344822,0.672997,19.338356
max,151.000000,4862.000000,20.740591,23.897542,2.416391,8.664866,1695.351298,3.414575,51.026407


In [29]:
# ------------------------------------------------------------
# CHECK HOW MANY MATCHES HAVE ENOUGH COMMON-OPPONENT HISTORY
# ------------------------------------------------------------

print("All matches:", len(intransitivity_df))

print(
    "Matches with at least 2 common opponents:",
    (intransitivity_df["num_common_opponents"] >= 2).sum()
)

print(
    "Matches with at least 3 local pair edges:",
    (intransitivity_df["num_local_edges"] >= 3).sum()
)

print(
    "Matches with nonzero I_star_direct:",
    (intransitivity_df["I_star_direct"] > 0).sum()
)

print(
    "Matches with nonzero I_star_context:",
    (intransitivity_df["I_star_context"] > 0).sum()
)

All matches: 15423
Matches with at least 2 common opponents: 13501
Matches with at least 3 local pair edges: 13501
Matches with nonzero I_star_direct: 5965
Matches with nonzero I_star_context: 13501


In [31]:
MAIN_INTRANSITIVITY_COL = "I_star_context"

intransitivity_df["intransitivity_level"] = "no_context_evidence"

valid_mask = intransitivity_df[MAIN_INTRANSITIVITY_COL] > 0

intransitivity_df.loc[valid_mask,
                      "intransitivity_level"
                      ] = pd.qcut(
                        intransitivity_df.loc[valid_mask, MAIN_INTRANSITIVITY_COL],
                        q = 3,
                        labels = ["low", "medium", "high"],
                        duplicates = "drop"
                      ).astype(str)

print(intransitivity_df["intransitivity_level"].value_counts(dropna=False))

intransitivity_level
low                    4501
medium                 4500
high                   4500
no_context_evidence    1922
Name: count, dtype: int64


In [32]:
valid_scores = intransitivity_df.loc[
    valid_mask,
    MAIN_INTRANSITIVITY_COL
]

low_threshold = valid_scores.quantile(1 / 3)
high_threshold = valid_scores.quantile(2 / 3)

print("Main score used:", MAIN_INTRANSITIVITY_COL)
print("Low/medium threshold:", low_threshold)
print("Medium/high threshold:", high_threshold)

Main score used: I_star_context
Low/medium threshold: 6.794886329107548
Medium/high threshold: 17.396678806394824


In [35]:
level_to_number = {
    "no_context_evidence": -1,
    "low": 0,
    "medium": 1,
    "high": 2
}

intransitivity_df["intransitivity_level_num"] = (
    intransitivity_df["intransitivity_level"]
    .map(level_to_number)
)

intransitivity_df.to_csv(
    PROCESSED_DIR / "match_intransitivity_scores_with_levels.csv",
    index=False
)

print("Saved match_intransitivity_scores_with_levels.csv")

Saved match_intransitivity_scores_with_levels.csv


In [36]:
bscore_matches["tourney_date"] = pd.to_datetime(
    bscore_matches["tourney_date"]
)

intransitivity_df["tourney_date"] = pd.to_datetime(
    intransitivity_df["tourney_date"]
)


# These columns identify the same match in both dataframes.
merge_keys = [
    "tourney_id",
    "tourney_date",
    "round",
    "surface",
    "winner_name",
    "loser_name"
]


# We do not need to merge every duplicated column from intransitivity_df.
# Keep only the columns that describe the score and the label.
intransitivity_cols_to_add = merge_keys + [
    "num_common_opponents",
    "num_local_edges",
    "norm_transitive",
    "norm_cyclic",
    "I_raw",
    "direct_evidence",
    "context_evidence",
    "I_star_direct",
    "I_star_context",
    "intransitivity_level",
    "intransitivity_level_num"
]


bscore_with_intransitivity = bscore_matches.merge(
    intransitivity_df[intransitivity_cols_to_add],
    on=merge_keys,
    how="left"
)


# Check whether the merge worked.
print("Original B-score dataset rows:", len(bscore_matches))
print("Merged dataset rows:", len(bscore_with_intransitivity))

print(
    "Rows without intransitivity level:",
    bscore_with_intransitivity["intransitivity_level"].isna().sum()
)

Original B-score dataset rows: 15423
Merged dataset rows: 15423
Rows without intransitivity level: 0


In [37]:
bscore_with_intransitivity["intransitivity_level"] = (
    bscore_with_intransitivity["intransitivity_level"]
    .fillna("missing")
)

bscore_with_intransitivity["intransitivity_level_num"] = (
    bscore_with_intransitivity["intransitivity_level_num"]
    .fillna(-2)
    .astype(int)
)


# Check final distribution.
print(
    bscore_with_intransitivity["intransitivity_level"]
    .value_counts(dropna=False)
)

intransitivity_level
low                    4501
medium                 4500
high                   4500
no_context_evidence    1922
Name: count, dtype: int64


In [38]:
bscore_with_intransitivity.to_csv(
    PROCESSED_DIR / "atp_matches_with_bscore_and_intransitivity.csv",
    index=False
)

print("Saved atp_matches_with_bscore_and_intransitivity.csv")

Saved atp_matches_with_bscore_and_intransitivity.csv


In [39]:
low_matches = bscore_with_intransitivity[
    bscore_with_intransitivity["intransitivity_level"] == "low"
].copy()

medium_matches = bscore_with_intransitivity[
    bscore_with_intransitivity["intransitivity_level"] == "medium"
].copy()

high_matches = bscore_with_intransitivity[
    bscore_with_intransitivity["intransitivity_level"] == "high"
].copy()


print("Low intransitivity matches:", len(low_matches))
print("Medium intransitivity matches:", len(medium_matches))
print("High intransitivity matches:", len(high_matches))


low_matches.to_csv(
    PROCESSED_DIR / "atp_matches_with_bscore_low_intransitivity.csv",
    index=False
)

medium_matches.to_csv(
    PROCESSED_DIR / "atp_matches_with_bscore_medium_intransitivity.csv",
    index=False
)

high_matches.to_csv(
    PROCESSED_DIR / "atp_matches_with_bscore_high_intransitivity.csv",
    index=False
)

Low intransitivity matches: 4501
Medium intransitivity matches: 4500
High intransitivity matches: 4500
